## Eftychia Kazasi - 8/9/2026

### Step 1: Load the dataset

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.special import gammaln

birds = pd.read_csv("/.../bird_count.csv") # adjust the path to your file

count = birds["count"].to_numpy(dtype=float) # response variable
year = birds["yr"].to_numpy(dtype=float) # predictor variable

- Poisson regression model with year as the predictor

Since we only have one predictor in this model, the equations describing it are the following:

$count \mid year_{i} \sim \text{Po}(\lambda(year_{i}))$ and $\log(\lambda(year_{i})) = \beta_0 + \beta_1 \cdot year_{i}$

So, the bird count in a given year is Poisson-distributed, and the log of its expected value changes linearly with year (link function).

### Step 2: Estimate the parameters using maximum likelihood

Based on the lecture notes, the log likelihood function for the Poisson GLM is:

$$\log l(y \mid \beta_0, \beta_1, \dots, \beta_p, x) = \sum_{i=1}^{n} \left[ -\lambda(x_i) + y_i \log(\lambda(x_i)) - \log(y_i!) \right] = \sum_{i=1}^{n} \left[ -e^{\beta_0 + \beta_1 x_{i1} + \dots + \beta_p x_{ip}} + y_i(\beta_0 + \beta_1 x_{i1} + \dots + \beta_p x_{ip}) - \log(y_i!) \right]$$

that in this example it changes into:

$$\log l(\text{count} \mid \beta_0, \beta_1, \text{year}) = \sum_{i=1}^{n} \left[ -\lambda(\text{year}_i) + \text{count}_i \log(\lambda(\text{year}_i)) - \log(\text{count}_i!) \right] = \sum_{i=1}^{n} \left[ -e^{\beta_0 + \beta_1 \text{year}_i} + \text{count}_i(\beta_0 + \beta_1 \text{year}_i) - \log(\text{count}_i!) \right]$$



In [ ]:
def log_likelihood(beta, year, count):

    beta0, beta1 = beta

    lt = beta0 + beta1 * year        # the linear term
    ex = np.exp(lt)             
    ll = np.sum(-ex + count * lt - gammaln(count + 1))
    return ll                     

*np.log(math.factorial(count)) can't take an array for input as the count variable is. So, I replace this term with this gammaln(count + 1), which is equal with calculating the log of the factorial.*

### Step 3: Optimization

In order to estimate the parameters using maximum likelihood, we should optimize the **log_likelihood** function. To do that, I will use a built in optimizer function from the scipy package, named **scipy.optimize.minimize**. I couldn't find any built in functions to maximize the log of the likelihood function ($$\hat{\theta} = \underset{\theta}{\operatorname{argmax}} \log l(\theta)$$, as mentioned in the notes), so we can just use the scipy function to minimize the negative log-likelihood, which is equivalent to maximising the log-likelihood.

In [ ]:
# helper function to return the negative log-likelihood that will be used for optimization through the minimize function

def neg_log_likelihood(beta, year, count):
    return -log_likelihood(beta, year, count)

As $β_0$ and $β_1$ initial values, that the optimizer requires, I set $β_1$ = 0 and now the model isn't depending on the year at all. So now, the average of the data is the best estimate of the underlying rate. The link function for Poisson GLM is $\log(\lambda(year_{i})) = \beta_0 + \beta_1 \cdot year_{i}$ as we said earlier, so, since the best guess for the rate ($β_0$) is the mean of all counts, the best guess for $\log(\lambda)$ is the log of the $$\mu(count)$$ 

In [32]:
beta_start = np.array([np.log(count.mean()), 0.0]) # starting values for beta0 and beta1

result = minimize(neg_log_likelihood, beta_start, args=(year, count), method="Nelder-Mead") # optimize the log-likelihood by minimizing the negative log-likelihood using the Nelder-Mead optimization method

beta0_est, beta1_est = result.x # the best estimates of beta0 and beta1 that best explain the observed data, based on maximum likelihood estimation
print("Convergence:", result.success) # sanity check of whether the optimizer successfully convergence


print(f"beta0_est = {beta0_est:.5f}")
print(f"beta1_est = {beta1_est:.5f}")

Convergence: True
beta0_est = 67.17918
beta1_est = -0.03244


- Convergence: True
- beta0_est = 67.17918
- beta1_est = -0.03244

### Step 4: Point estimate predictions from the time period 

In [29]:
all_years = np.arange(year.min(), year.max() + 1) # an array with all years between 1999 and 2012, to make predictions for every year in that range

lambda_est = np.exp(beta0_est + beta1_est * all_years) # compute the expected counts for each year using the estimated parameters beta0_est and beta1_est

### Step 5: Generate 3 samples of data from the optimized model

Use the fitted model to simulate hypothetical bird counts. For each year, in order to generate random samples for the 1999-2012 time period, we can use the *lamda_est* we calculated before based on the optimized parameters and pick a random count from a $$\text{Poisson}(\hat{\lambda}_{\text{year}})$$ distribution. We repeat this step 3 separate times .

In [ ]:
np.random.seed(42)  # use the same seed for reproducibility

sample_tables = []

for s in range(1, 4): # repeat three times
    simulated_counts = np.random.poisson(lam=lambda_est) # one random count for each year picked from the Poisson distribution

    sample_tables.append(pd.DataFrame({
        "sample": s,
        "year": all_years.astype(int),
        "lambda_est": lambda_est,
        "count": simulated_counts
    }))

samples_df = pd.concat(sample_tables, ignore_index=True)
samples_df.to_csv("/.../hypothetical_observations.csv", index=False) # adjust the path to your file